In [1]:
# ==========================================
# 1. IMPORTS & MOUNT DRIVE
# ==========================================
import os
import collections
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from tqdm import tqdm

# ==========================================
# 2. SET UP DEVICE & SAVE PATH
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device: {device}")

if device.type == 'cpu':
    raise SystemError("🚨 ERROR: Switch to T4 GPU in Colab before running!")

# NEW: Define exactly where to save the model
SAVE_DIR = '/content/drive/MyDrive/Colab Notebooks/R26-IT-012_All_Datasets/Trained_Models/'
os.makedirs(SAVE_DIR, exist_ok=True) # Creates the folder if it doesn't exist
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, 'efficientnet_gastric_best.pt')

# ==========================================
# 3. LOAD DATA (Required in memory for training)
# ==========================================
base_dir = '/content/drive/MyDrive/Colab Notebooks/R26-IT-012_All_Datasets/Gastric_Module_Images/labeled-images/upper-gi-tract'
target_classes = {'normal-z-line': 0, 'pylorus': 0, 'erythema': 1, 'esophagitis-a': 1, 'esophagitis-b-d': 1}

image_paths, labels = [], []
for root, dirs, files in os.walk(base_dir):
    folder_name = os.path.basename(root)
    if folder_name in target_classes:
        label = target_classes[folder_name]
        for file in files:
            if file.endswith(('.jpg', '.jpeg', '.png')):
                image_paths.append(os.path.join(root, file))
                labels.append(label)

class_counts = collections.Counter(labels)
total_samples = len(labels)
num_classes = 2
class_weights = torch.tensor([
    total_samples / (num_classes * class_counts[0]), 
    total_samples / (num_classes * class_counts[1])
], dtype=torch.float32)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

IMAGE_SIZE = 224 
train_transforms = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

class GastricDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        image = cv2.cvtColor(cv2.imread(self.image_paths[idx]), cv2.COLOR_BGR2RGB)
        if self.transform: image = self.transform(image=image)['image']
        return image, torch.tensor(self.labels[idx], dtype=torch.long)

BATCH_SIZE = 32
train_loader = DataLoader(GastricDataset(train_paths, train_labels, transform=train_transforms), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(GastricDataset(val_paths, val_labels, transform=val_transforms), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ==========================================
# 4. INITIALIZE MODEL & TRAIN
# ==========================================
model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=2).to(device)
class_weights = class_weights.to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

EPOCHS = 10
best_val_accuracy = 0.0

print("\n--- Starting Training ---")
for epoch in range(EPOCHS):
    # Train
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    train_acc = 100 * correct_train / total_train
    avg_train_loss = running_loss / len(train_loader)
    
    # Val
    model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    val_acc = 100 * correct_val / total_val
    avg_val_loss = val_loss / len(val_loader)
    
    print(f"\nEpoch [{epoch+1}/{EPOCHS}] Summary:")
    print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss:   {avg_val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    
    # Save directly to the specific Drive path
    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"🌟 New best model saved to Drive! Val Acc: {best_val_accuracy:.2f}%\n")

print(f"\n🎉 Training Complete! Model saved at: {MODEL_SAVE_PATH}")

Training on device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]


--- Starting Training ---


Epoch 1/10 [Val]: 100%|██████████| 11/11 [02:08<00:00, 11.71s/it]


...


Epoch [8/10] Summary:
Train Loss: 0.0527 | Train Acc: 98.15%
Val Loss:   0.0563 | Val Acc:   98.46%


Epoch 9/10 [Val]: 100%|██████████| 11/11 [00:08<00:00,  1.35it/s]



Epoch [9/10] Summary:
Train Loss: 0.0299 | Train Acc: 98.61%
Val Loss:   0.0627 | Val Acc:   97.84%


Epoch 10/10 [Val]: 100%|██████████| 11/11 [00:07<00:00,  1.39it/s]


Epoch [10/10] Summary:
Train Loss: 0.0313 | Train Acc: 99.15%
Val Loss:   0.0575 | Val Acc:   98.46%

🎉 Training Complete! Model saved at: /content/drive/MyDrive/Colab Notebooks/R26-IT-012_All_Datasets/Trained_Models/efficientnet_gastric_best.pt
